# Preprocessing And Feature Engineering

This notebook documents the supervised learning matrix and leakage controls. Features are based on information available before the target month.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Image, Markdown, display
except Exception:
    def display(value):
        print(value)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image(filename={self.filename!r})"

ROOT = Path.cwd()
if ROOT.name != "v7_rm_pm_forecast_planning":
    ROOT = Path("Ai miroservices/modeling/v7_rm_pm_forecast_planning").resolve()
OUT = ROOT / "outputs"
PLOTS = OUT / "plots"
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

def load_csv(name, **kwargs):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return pd.read_csv(path, **kwargs)

def load_json(name):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return json.loads(path.read_text())

def show_plot(name):
    path = PLOTS / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"Plot not generated: `{path}`"))

In [2]:
profile = load_csv('feature_matrix_profile.csv')
profile

,column,dtype,non_null_rows,null_rows,mean,std,min,max
0,month_num,int32,6624,0,6.739130,3.326267,1.0,12.000000
1,quarter,int32,6624,0,2.565217,1.096563,1.0,4.000000
2,year,int32,6624,0,2024.521739,0.499565,2024.0,2025.000000
3,month_sin,float64,6624,0,-0.021739,0.714475,-1.0,1.000000
4,month_cos,float64,6624,0,-0.037653,0.698417,-1.0,1.000000
5,material_type_enc,int8,6624,0,0.000000,0.000000,0.0,0.000000
6,material_code_enc,int16,6624,0,143.500000,83.144214,0.0,287.000000
7,lag_1,float64,6624,0,1876.042874,9057.073061,0.0,140237.000000
8,lag_2,float64,6624,0,1888.631643,9112.802546,0.0,140237.000000
9,lag_3,float64,6624,0,1887.064312,9106.798436,0.0,140237.000000


In [3]:
sample = load_csv("feature_matrix_sample.csv", parse_dates=["month"])
sample.head(30)

,material_id,material_code,description,material_type,warehouse_id,warehouse_code,demand_units,promotion_flag,holiday_flag,source,month,month_num,quarter,year,month_sin,month_cos,material_type_enc,material_code_enc,lag_1,lag_2,lag_3,lag_6,lag_12,roll_mean_3,roll_std_3,roll_mean_6,roll_std_6,roll_mean_12,roll_std_12,target
0,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,131.0,False,False,canonical_v6,2024-02-01,2,1,2024,8.660254e-01,5.000000e-01,0,3,123.0,140.0,150.0,130.0,124.0,137.666667,13.650397,120.833333,51.320236,139.416667,50.699576,4.0
1,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,4.0,False,False,canonical_v6,2024-03-01,3,1,2024,1.000000e+00,6.123234e-17,0,3,131.0,123.0,140.0,20.0,130.0,131.333333,8.504901,121.000000,51.357570,140.000000,50.546109,171.0
2,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,171.0,False,True,canonical_v6,2024-04-01,4,2,2024,8.660254e-01,-5.000000e-01,0,3,4.0,131.0,123.0,162.0,178.0,86.000000,71.126648,118.333333,57.677263,129.500000,64.085880,108.0
3,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,108.0,False,False,canonical_v6,2024-05-01,5,2,2024,5.000000e-01,-8.660254e-01,0,3,171.0,4.0,131.0,150.0,99.0,102.000000,87.195183,119.833333,59.138538,128.916667,63.634551,209.0
4,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,209.0,False,False,canonical_v6,2024-06-01,6,2,2024,1.224647e-16,-1.000000e+00,0,3,108.0,171.0,4.0,140.0,213.0,94.333333,84.334651,112.833333,57.311139,129.666667,63.302066,267.0
5,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,267.0,False,False,canonical_v6,2024-07-01,7,3,2024,-5.000000e-01,-8.660254e-01,0,3,209.0,108.0,171.0,123.0,204.0,162.666667,51.013070,124.333333,69.482852,129.333333,62.832148,107.0
6,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,107.0,False,False,canonical_v6,2024-08-01,8,3,2024,-8.660254e-01,-5.000000e-01,0,3,267.0,209.0,108.0,131.0,130.0,194.666667,80.463242,148.333333,90.592862,134.583333,71.651249,208.0
7,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,208.0,False,False,canonical_v6,2024-09-01,9,3,2024,-1.000000e+00,-1.836970e-16,0,3,107.0,267.0,209.0,4.0,20.0,194.333333,81.002058,144.333333,92.029705,132.666667,72.091272,135.0
8,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,135.0,False,False,canonical_v6,2024-10-01,10,4,2024,-8.660254e-01,5.000000e-01,0,3,208.0,107.0,267.0,171.0,162.0,194.000000,80.913534,178.333333,62.882960,148.333333,65.508269,100.0
9,00c543ff-b1f0-4938-a6ac-e490ea98b94d,100012,AMPHISOL,raw_material,7262019d-9bf4-4824-997c-d7b5c9158ef3,WH-001,100.0,False,True,canonical_v6,2024-11-01,11,4,2024,-5.000000e-01,8.660254e-01,0,3,135.0,208.0,107.0,108.0,150.0,150.000000,52.144031,172.333333,65.390111,146.083333,65.459854,115.0


In [4]:
sample[["month", "target", "lag_1", "lag_2", "lag_3", "lag_6", "lag_12", "roll_mean_3", "roll_mean_6", "roll_mean_12"]].head(20)

,month,target,lag_1,lag_2,lag_3,lag_6,lag_12,roll_mean_3,roll_mean_6,roll_mean_12
0,2024-02-01,4.0,123.0,140.0,150.0,130.0,124.0,137.666667,120.833333,139.416667
1,2024-03-01,171.0,131.0,123.0,140.0,20.0,130.0,131.333333,121.000000,140.000000
2,2024-04-01,108.0,4.0,131.0,123.0,162.0,178.0,86.000000,118.333333,129.500000
3,2024-05-01,209.0,171.0,4.0,131.0,150.0,99.0,102.000000,119.833333,128.916667
4,2024-06-01,267.0,108.0,171.0,4.0,140.0,213.0,94.333333,112.833333,129.666667
5,2024-07-01,107.0,209.0,108.0,171.0,123.0,204.0,162.666667,124.333333,129.333333
6,2024-08-01,208.0,267.0,209.0,108.0,131.0,130.0,194.666667,148.333333,134.583333
7,2024-09-01,135.0,107.0,267.0,209.0,4.0,20.0,194.333333,144.333333,132.666667
8,2024-10-01,100.0,208.0,107.0,267.0,171.0,162.0,194.000000,178.333333,148.333333
9,2024-11-01,115.0,135.0,208.0,107.0,108.0,150.0,150.000000,172.333333,146.083333


In [5]:
leakage_rules = pd.DataFrame([
    {"feature_group": "lags", "rule": "lag_n uses demand from n months before the feature row month"},
    {"feature_group": "rolling means/std", "rule": "rolling windows are shifted by 1 before aggregation"},
    {"feature_group": "calendar", "rule": "month, quarter, year are known before forecast generation"},
    {"feature_group": "material encodings", "rule": "material code/type are static metadata"},
    {"feature_group": "target", "rule": "target is next-month demand and is never used as an input feature"},
])
leakage_rules

,feature_group,rule
0,lags,lag_n uses demand from n months before the fea...
1,rolling means/std,rolling windows are shifted by 1 before aggreg...
2,calendar,"month, quarter, year are known before forecast..."
3,material encodings,material code/type are static metadata
4,target,target is next-month demand and is never used ...
